In [1]:
import xarray as xr
import glob

files = sorted(glob.glob("/scratch/benbar/NAARC/Initial_Conditions/IC_compressed_*.nc"))
datasets = [xr.open_dataset(f, chunks={}) for f in files]

In [2]:
datasets[0]

<xarray.Dataset> Size: 1GB
Dimensions:  (z: 75, y: 543, x: 3240)
Dimensions without coordinates: z, y, x
Data variables:
    toce     (z, y, x) float32 528MB dask.array<chunksize=(1, 272, 1080), meta=np.ndarray>
    soce     (z, y, x) float32 528MB dask.array<chunksize=(1, 272, 1080), meta=np.ndarray>

In [3]:
rows = []

for iy in range(4):
    row = xr.concat(
        [datasets[iy * 4 + ix] for ix in range(4)],
        dim="x"
    )
    rows.append(row)

ds_full = xr.concat(rows, dim="y")

In [4]:
ds_full

<xarray.Dataset> Size: 17GB
Dimensions:  (z: 75, y: 2173, x: 12960)
Dimensions without coordinates: z, y, x
Data variables:
    toce     (z, y, x) float32 8GB dask.array<chunksize=(1, 272, 1080), meta=np.ndarray>
    soce     (z, y, x) float32 8GB dask.array<chunksize=(1, 272, 1080), meta=np.ndarray>

In [5]:
ds_full = ds_full.chunk({'z': 3, 'y': 544, 'x': 3240})

In [6]:
fill_value = 1e20
ds_full = ds_full.fillna(fill_value)

encoding = {
    var: {
        "zlib": True,
        "complevel": 1,
        "_FillValue": fill_value,
        "dtype": "float32",
        "chunksizes": (3, 544, 3240),  # (deptht, y, x) chunk shape on disk
    }
    for var in ds_full.data_vars
}

ds_full.to_netcdf(
    "/scratch/benbar/NAARC/Initial_Conditions/IC_compressed.nc",
    mode="w",
    format="NETCDF4",
    engine="netcdf4",
    encoding=encoding,
)